In [1]:
# STEP 1: Import libraries and define file paths
# Description: Load required Python libraries and set all file paths

import pandas as pd
import os

# Input EPC dataset
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_D2_windows.csv"

# Lighting efficiency lookup table
lighting_eff_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/lighting_efficiency.csv"

# Output file path
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_lighting_watts.csv"

In [2]:
# STEP 2: Load datasets
# Description: Read EPC dataset and lighting efficiency lookup table into pandas DataFrames

epc_df = pd.read_csv(epc_path)
lighting_eff_df = pd.read_csv(lighting_eff_path)

# Preview data (optional for debugging)
print(epc_df.head())
print(lighting_eff_df.head())

   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
3  FINTRY   GLASGOW   G63 0YJ  56.058427  -4.224838         7/22/25   
4  FINTRY   GLASGOW   G63 0XF  56.054738  -4.228307         7/17/25   

         TYPE_OF_ASSESSMENT  ...  \
0  RdSAP, existing dwelling  ...   
1  RdSAP, existing dwelling  ...   
2  RdSAP

In [3]:
# STEP 3: Clean and standardise column names
# Description: Ensure consistent column naming for merging

# Standardise column names (strip whitespace and uppercase)
epc_df.columns = epc_df.columns.str.strip().str.upper()
lighting_eff_df.columns = lighting_eff_df.columns.str.strip().str.upper()

# Rename columns in lookup table if necessary
lighting_eff_df = lighting_eff_df.rename(columns={
    "GRADE": "LIGHTING_ENERGY_EFF",
    "LUMENS/WATT": "LIGHTING_LUM_PER_W"
})

# Ensure no trailing spaces in categorical values
epc_df["LIGHTING_ENERGY_EFF"] = epc_df["LIGHTING_ENERGY_EFF"].str.strip()
lighting_eff_df["LIGHTING_ENERGY_EFF"] = lighting_eff_df["LIGHTING_ENERGY_EFF"].str.strip()

In [4]:
# STEP 4: Merge EPC data with lighting efficiency lookup
# Description: Map lighting grades to lumens per watt values

epc_df = epc_df.merge(
    lighting_eff_df[["LIGHTING_ENERGY_EFF", "LIGHTING_LUM_PER_W"]],
    on="LIGHTING_ENERGY_EFF",
    how="left"
)

# Check for missing mappings
missing = epc_df["LIGHTING_LUM_PER_W"].isna().sum()
print(f"Missing lighting efficiency values: {missing}")

Missing lighting efficiency values: 0


In [5]:
# STEP 5: Calculate lighting lumens
# Description: Compute total lumens based on floor area

# Ensure floor area is numeric
epc_df["TOTAL_FLOOR_AREA"] = pd.to_numeric(epc_df["TOTAL_FLOOR_AREA"], errors="coerce")

# Lighting lumens = floor area * 186
epc_df["LIGHTING_LUMENS"] = epc_df["TOTAL_FLOOR_AREA"] * 186

In [6]:
# STEP 6: Calculate lighting watts
# Description: Compute total lighting wattage

epc_df["LIGHTING_WATTS"] = epc_df["LIGHTING_LUMENS"] / epc_df["LIGHTING_LUM_PER_W"]

# Optional: round values for readability
epc_df["LIGHTING_LUM_PER_W"] = epc_df["LIGHTING_LUM_PER_W"].round(2)
epc_df["LIGHTING_LUMENS"] = epc_df["LIGHTING_LUMENS"].round(2)
epc_df["LIGHTING_WATTS"] = epc_df["LIGHTING_WATTS"].round(2)

In [7]:
# STEP 7: Save output dataset
# Description: Export the updated EPC dataset with new lighting columns

epc_df.to_csv(output_path, index=False)

print(f"Updated dataset saved to: {output_path}")

Updated dataset saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_lighting_watts.csv


In [8]:
# STEP 8: (Optional) Quick validation checks
# Description: Sanity check calculations

print(epc_df[[
    "BUILDING_REFERENCE_NUMBER",
    "LIGHTING_ENERGY_EFF",
    "LIGHTING_LUM_PER_W",
    "TOTAL_FLOOR_AREA",
    "LIGHTING_LUMENS",
    "LIGHTING_WATTS"
]].head())

   BUILDING_REFERENCE_NUMBER LIGHTING_ENERGY_EFF  LIGHTING_LUM_PER_W  \
0                 1001829067                Good                66.9   
1                 1001951858                Good                66.9   
2                 1001829071           Very Good                80.5   
3                 1234761001                Good                66.9   
4                 1001991633                Good                66.9   

   TOTAL_FLOOR_AREA  LIGHTING_LUMENS  LIGHTING_WATTS  
0                42             7812          116.77  
1               126            23436          350.31  
2                68            12648          157.12  
3               127            23622          353.09  
4               104            19344          289.15  
